# Feed the Beast : démarrage sur Colab / Kaggle

Exécution > Modifier le type d'exécution > **GPU T4**. Ce notebook fait tourner toute la chaîne avec la baseline d'exemple : remplace ensuite `example_data/` par **tes** données.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!git clone -q https://github.com/Meuleur/Piscine.git
%cd Piscine
!pip install -q -r requirements.txt

## (Optionnel) Sauvegarder les runs sur Google Drive

Une session Colab peut couper. Avec les runs sur Drive, `--resume` reprend là où l'entraînement s'est arrêté.

In [ ]:
RUNS = 'runs'
# from google.colab import drive; drive.mount('/content/drive')
# RUNS = '/content/drive/MyDrive/feed-the-beast/runs'

## 1. Données d'exemple (baseline naïve, ~450 Mo)

In [ ]:
!python examples/make_example_dataset.py --out example_data/ --mo 450

## 2. Tokenizer (8 192 tokens) et flux de tokens

In [ ]:
!python -m ftb.tokenizer train --inputs example_data/example.jsonl --out tokenizer.json
!python -m ftb.prepare --inputs example_data/example.jsonl --tokenizer tokenizer.json --out prepared/

## 3. Entraînement officiel (100M tokens)

Le script affiche le débit et le temps restant. Ajoute `--compile` sur T4 pour aller plus vite (pas sur P100). Si la session coupe, relance la même cellule : `--resume` repart du dernier point de sauvegarde.

In [ ]:
!python -m ftb.train --data prepared/ --tokenizer tokenizer.json --out {RUNS}/baseline --resume

## 4. Évaluation et échantillons

In [ ]:
!python -m ftb.evaluate --ckpt {RUNS}/baseline/ckpt.pt --tokenizer {RUNS}/baseline/tokenizer.json
!cat {RUNS}/baseline/samples.txt

## 5. Soumettre

La soumission se pousse depuis **ton** dépôt : télécharge le dossier `submissions/...` créé ci-dessous (ou lance la commande depuis ton dépôt), puis `git add`, `git commit`, `git push`.

In [ ]:
!python -m ftb.submit --run {RUNS}/baseline --participant "Prénom Nom"